# 🚗 Object Detection for Self-Driving using YOLO
## Project Code: SIC/AI/003
**Team:** Naman Kumar | Aryan Kaushik | Muskan | Adhisha Pahuja | Sukesh Bhardwaj

---
This notebook covers the **complete pipeline**:
1. ✅ Environment Setup
2. ✅ Dataset Download & Preprocessing
3. ✅ Model Training (YOLOv8)
4. ✅ Evaluation & Metrics
5. ✅ Real-time Inference Demo
6. ✅ Model Export

> **Tip:** Go to `Runtime → Change runtime type → GPU (T4)` for free GPU!

## 📦 Step 1: Environment Setup

In [ ]:
# ── Check GPU availability ──
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Install dependencies ──
!pip install ultralytics roboflow albumentations wandb seaborn -q

# Verify installation
from ultralytics import YOLO
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

In [ ]:
# ── Clone project repository (update URL with your GitHub repo) ──
!git clone https://github.com/YOUR_USERNAME/yolo_selfdriving.git
%cd yolo_selfdriving

# OR: Upload project zip
# from google.colab import files
# uploaded = files.upload()
# !unzip yolo_selfdriving.zip

## 🗂️ Step 2: Dataset Setup

In [ ]:
# ── OPTION A: Use Roboflow dataset (Recommended for quick start) ──
# Sign up free at roboflow.com to get your API key

from roboflow import Roboflow

rf = Roboflow(api_key='YOUR_ROBOFLOW_API_KEY')  # Replace with your key

# Option 1: Self-driving dataset
project = rf.workspace('public').project('self-driving-car-gqp0s')
dataset = project.version(3).download('yolov8')

print('Dataset downloaded to:', dataset.location)
DATASET_YAML = dataset.location + '/data.yaml'

In [ ]:
# ── OPTION B: Use COCO dataset subset (already in Ultralytics) ──
# This downloads a small 8-image subset for testing

DATASET_YAML = 'coco8.yaml'  # Built-in COCO 8-image subset

# For full COCO:
# DATASET_YAML = 'coco.yaml'

print('Using dataset:', DATASET_YAML)

In [ ]:
# ── OPTION C: Use our custom dataset.yaml ──
import os

# Mount Google Drive (if dataset is stored there)
from google.colab import drive
drive.mount('/content/drive')

# Set path to your dataset yaml in Drive
DATASET_YAML = '/content/drive/MyDrive/SIC_AI_003/dataset.yaml'
print('Dataset YAML:', DATASET_YAML)

In [ ]:
# ── Run dataset preprocessing (if using our custom script) ──
import sys
sys.path.insert(0, './src')

from dataset import setup_directories, dataset_statistics

setup_directories('./data')
dataset_statistics('./data')

## 🚀 Step 3: Model Training

In [ ]:
# ── Configure training parameters ──

MODEL_SIZE = 'yolov8m.pt'   # yolov8n/s/m/l/x.pt  (n=fastest, x=most accurate)
EPOCHS     = 50             # Start with 50 for quick results; use 100-200 for production
BATCH_SIZE = 16             # Reduce to 8 if GPU runs out of memory
IMG_SIZE   = 640            # Standard YOLO input size
DEVICE     = 0              # GPU device ID (0 for first GPU)

print(f'Model    : {MODEL_SIZE}')
print(f'Epochs   : {EPOCHS}')
print(f'Batch    : {BATCH_SIZE}')
print(f'Img Size : {IMG_SIZE}×{IMG_SIZE}')
print(f'Dataset  : {DATASET_YAML}')

In [ ]:
# ── TRAIN THE MODEL ──
from ultralytics import YOLO

# Load pretrained YOLOv8 weights (fine-tuning on our dataset)
model = YOLO(MODEL_SIZE)

# Start training
results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    device    = DEVICE,
    optimizer = 'AdamW',
    lr0       = 0.01,
    lrf       = 0.001,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    mosaic    = 1.0,
    fliplr    = 0.5,
    mixup     = 0.1,
    patience  = 15,
    save      = True,
    save_period = 10,
    plots     = True,
    project   = './runs/train',
    name      = 'sic_ai_003',
    verbose   = True,
)

print('\n✅ Training Complete!')
print('Best mAP@0.5:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

In [ ]:
# ── Visualize training curves ──
import pandas as pd
import matplotlib.pyplot as plt

results_csv = './runs/train/sic_ai_003/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Training Curves — SIC/AI/003', fontsize=14, fontweight='bold')

plots = [
    ('train/box_loss',      'Train Box Loss',    'royalblue'),
    ('train/cls_loss',      'Train Class Loss',  'darkorange'),
    ('train/dfl_loss',      'Train DFL Loss',    'seagreen'),
    ('metrics/mAP50(B)',    'mAP @ 0.5',         'crimson'),
    ('metrics/mAP50-95(B)', 'mAP @ 0.5:0.95',   'purple'),
    ('val/box_loss',        'Val Box Loss',       'saddlebrown'),
]

for ax, (col, title, color) in zip(axes.flatten(), plots):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=2)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.35)

plt.tight_layout()
plt.savefig('./results/training_curves.png', dpi=150)
plt.show()

## 📊 Step 4: Model Evaluation

In [ ]:
# ── Load best trained weights & evaluate ──
best_model_path = './runs/train/sic_ai_003/weights/best.pt'
model = YOLO(best_model_path)

# Run evaluation on validation set
metrics = model.val(
    data  = DATASET_YAML,
    split = 'val',
    conf  = 0.001,    # Low conf for full precision-recall curve
    iou   = 0.60,
    plots = True,
    verbose = True,
)

# Print summary
print('\n' + '='*50)
print('  EVALUATION RESULTS — SIC/AI/003')
print('='*50)
print(f'  mAP@0.5       : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95  : {metrics.box.map:.4f}')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')
f1 = 2*metrics.box.mp*metrics.box.mr/(metrics.box.mp+metrics.box.mr+1e-10)
print(f'  F1 Score      : {f1:.4f}')
print('='*50)

In [ ]:
# ── Per-class metrics visualization ──
import numpy as np

CLASS_NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'autorickshaw',
    'bus', 'truck', 'traffic_light', 'stop_sign',
    'speed_breaker', 'pothole', 'animal', 'construction'
]

if hasattr(metrics.box, 'ap_class_index'):
    names, ap50s, precs, recs = [], [], [], []
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        names.append(CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'cls_{cls_idx}')
        ap50s.append(metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0)
        precs.append(metrics.box.p[i]    if i < len(metrics.box.p)    else 0)
        recs.append(metrics.box.r[i]     if i < len(metrics.box.r)    else 0)

    x     = np.arange(len(names))
    width = 0.27

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.bar(x - width, ap50s, width, label='AP@0.5',   color='#2E75B6', alpha=0.85)
    ax.bar(x,         precs, width, label='Precision', color='#70AD47', alpha=0.85)
    ax.bar(x + width, recs,  width, label='Recall',    color='#ED7D31', alpha=0.85)
    ax.axhline(0.75, color='red', linestyle='--', lw=1.2, label='Target (0.75)')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.set_title('Per-Class Metrics — SIC/AI/003', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.35)
    plt.tight_layout()
    plt.savefig('./results/per_class_metrics.png', dpi=150)
    plt.show()

In [ ]:
# ── Speed Benchmark ──
import time
import numpy as np

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(10):
    model(dummy, verbose=False)

# Measure
latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    model(dummy, verbose=False)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f'Mean Latency : {latencies.mean():.2f} ms')
print(f'Std Dev      : {latencies.std():.2f} ms')
print(f'Mean FPS     : {1000/latencies.mean():.1f}')
print(f'Target ≥30fps: {"✅ PASS" if 1000/latencies.mean() >= 30 else "❌ FAIL"}')

## 🎥 Step 5: Real-Time Inference Demo

In [ ]:
# ── Run inference on a test image ──
import cv2
import requests
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from io import BytesIO

CLASS_COLORS = [
    (0,255,0),(255,128,0),(0,128,255),(255,0,128),(128,0,255),
    (0,255,255),(255,255,0),(0,0,255),(50,50,200),(128,128,0),
    (0,128,128),(0,80,180),(180,180,180)
]

def run_detection(model, img_path_or_url, conf=0.35):
    """Detect objects and display annotated result."""
    # Load image
    if img_path_or_url.startswith('http'):
        r = requests.get(img_path_or_url)
        img = np.array(Image.open(BytesIO(r.content)).convert('RGB'))
        frame = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        frame = cv2.imread(img_path_or_url)

    results = model(frame, conf=conf, verbose=False)[0]

    for box in results.boxes:
        cls_id = int(box.cls)
        conf_s = float(box.conf)
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
        name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        label = f'{name}: {conf_s:.2f}'
        cv2.putText(frame, label, (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    # Display in Colab
    display_img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(display_img)
    plt.title(f'Detections: {len(results.boxes)} — SIC/AI/003', fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return frame

# Test with a sample driving image
SAMPLE_URL = 'https://ultralytics.com/images/bus.jpg'
run_detection(model, SAMPLE_URL, conf=0.35)

In [ ]:
# ── Run detection on a video file ──
# Upload your video first or use a sample
from google.colab import files
print('Upload a video file:')
# uploaded = files.upload()  # Uncomment to upload

# Or use built-in Ultralytics video prediction:
VIDEO_PATH = 'your_video.mp4'  # Change this

# model.predict(
#     source=VIDEO_PATH,
#     conf=0.35,
#     iou=0.45,
#     save=True,
#     project='./results',
#     name='video_output'
# )
print('Video detection: uncomment and set VIDEO_PATH to run')

## 📤 Step 6: Model Export

In [ ]:
# ── Export to ONNX (for cross-platform deployment) ──
best_model_path = './runs/train/sic_ai_003/weights/best.pt'
model = YOLO(best_model_path)

# Export to ONNX
onnx_path = model.export(format='onnx', imgsz=640, optimize=True)
print('ONNX model:', onnx_path)

# Export to TorchScript
ts_path = model.export(format='torchscript', imgsz=640)
print('TorchScript model:', ts_path)

# For edge devices (Raspberry Pi, Jetson):
# model.export(format='tflite', imgsz=640)  # TensorFlow Lite
# model.export(format='engine', imgsz=640)  # TensorRT (requires NVIDIA GPU)

In [ ]:
# ── Download trained model to local machine ──
from google.colab import files
import shutil

# Download best weights
files.download('./runs/train/sic_ai_003/weights/best.pt')

# OR: Copy to Google Drive
# shutil.copy('./runs/train/sic_ai_003/weights/best.pt',
#             '/content/drive/MyDrive/SIC_AI_003/best.pt')
# print('Saved to Google Drive!')

print('✅ Download complete!')

---
## 🏁 Summary

| Step | Status |
|------|--------|
| Environment Setup | ✅ |
| Dataset Loaded | ✅ |
| Model Trained | ✅ |
| Evaluation Complete | ✅ |
| Demo Inference | ✅ |
| Model Exported | ✅ |

**Project Code:** SIC/AI/003  
**Team:** Naman Kumar | Aryan Kaushik | Muskan | Adhisha Pahuja | Sukesh Bhardwaj
